# Combined Detection - Head + Eyes + Body

Run cells 1-4 in order. Press 'q' to quit.

In [1]:
# CELL 1: Load models
import cv2, time, numpy as np, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

face_landmarker = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path='../01_head_direction/face_landmarker.task'),
    num_faces=1, min_face_detection_confidence=0.5, min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5, running_mode=vision.RunningMode.IMAGE))

pose_landmarker = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path='pose_landmarker.task'),
    num_poses=1, min_pose_detection_confidence=0.5, min_tracking_confidence=0.5,
    running_mode=vision.RunningMode.IMAGE))

print('Models loaded!')

Models loaded!


In [2]:
# CELL 2: Configuration
HEAD_TURN_THRESHOLD = 0.15
HEAD_UP_THRESHOLD = 0.1
GAZE_THRESHOLD = 0.12       # WAS 0.25 — now MORE sensitive to eye peeks
BODY_TURN_THRESHOLD = 0.06

HEAD_WEIGHT = 70
EYE_WEIGHT = 60
BODY_WEIGHT = 50

MILD_THRESHOLD = 25
SUSPICIOUS_THRESHOLD = 45
HIGH_THRESHOLD = 65

HISTORY_SIZE = 300          # WAS 100 — longer history = center doesn't shift easily
MIN_READINGS = 30           # WAS 15 — more stable before detecting

print(f'Config set! Head:{HEAD_WEIGHT} Eyes:{EYE_WEIGHT} Body:{BODY_WEIGHT}')
print(f'Gaze threshold: {GAZE_THRESHOLD} (was 0.25, now 0.12 = catches small peeks)')
print(f'History: {HISTORY_SIZE} readings (longer = center stays stable)')

Config set! Head:70 Eyes:60 Body:50
Gaze threshold: 0.12 (was 0.25, now 0.12 = catches small peeks)
History: 300 readings (longer = center stays stable)


In [3]:
# CELL 3: Detection functions
def detect_head(fl, w, h):
    nose_x = fl[4].x * w
    eye_cx = (fl[133].x * w + fl[362].x * w) / 2
    eye_cy = (fl[133].y + fl[362].y) / 2 * h
    eye_dist = abs(fl[362].x * w - fl[133].x * w)
    if eye_dist == 0: return 'UNKNOWN', 0
    hr = (nose_x - eye_cx) / eye_dist
    vr = (fl[4].y * h - eye_cy) / eye_dist
    if vr < -HEAD_UP_THRESHOLD: d = 'UP'
    elif hr < -HEAD_TURN_THRESHOLD: d = 'LEFT'
    elif hr > HEAD_TURN_THRESHOLD: d = 'RIGHT'
    else: return 'FORWARD', 0
    return d, min(HEAD_WEIGHT, int((max(abs(hr), abs(vr)) / 0.4) * HEAD_WEIGHT))

def detect_eyes(fl, w, h, center):
    lw = abs(fl[133].x - fl[33].x); rw = abs(fl[362].x - fl[263].x)
    lr = (fl[468].x - fl[33].x) / lw if lw > 0 else 0.5
    rr = (fl[473].x - fl[263].x) / rw if rw > 0 else 0.5
    avg = (lr + rr) / 2; dev = avg - center
    if abs(dev) < GAZE_THRESHOLD: return 'CENTER', 0, avg
    d = 'LEFT' if dev > 0 else 'RIGHT'
    return d, min(EYE_WEIGHT, int((abs(dev) / 0.4) * EYE_WEIGHT)), avg

def detect_body(pl, w, h):
    yd = pl[12].y - pl[11].y
    vd = 0
    if hasattr(pl[11], 'visibility') and hasattr(pl[12], 'visibility'):
        vd = pl[11].visibility - pl[12].visibility
    rot = abs(yd) + abs(vd) * 0.3
    if rot < BODY_TURN_THRESHOLD: return 'FORWARD', 0
    d = 'LEFT' if yd > 0 else 'RIGHT'
    return d, min(BODY_WEIGHT, int((rot / 0.15) * BODY_WEIGHT))

print('Functions ready!')

Functions ready!


In [4]:
# CELL 4: Live detection - press 'q' to quit
cam = cv2.VideoCapture(0)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cam.isOpened():
    print('Cannot open webcam!')
else:
    print('Running! Press q to quit.')
    gh = []; ec = 0.5; fc = 0; t0 = time.time()

    while True:
        ok, f = cam.read()
        if not ok: break
        f = cv2.flip(f, 1); fc += 1
        h, w = f.shape[:2]
        rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        mi = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        fr = face_landmarker.detect(mi)
        pr = pose_landmarker.detect(mi)

        hs=0; es=0; bs=0; ht='Head:--'; et='Eyes:--'; bt='Body:--'

        if fr.face_landmarks and len(fr.face_landmarks)>0:
            fl=fr.face_landmarks[0]
            hd,hs=detect_head(fl,w,h); ht=f'Head:{hd}'
            ed,es,av=detect_eyes(fl,w,h,ec); gh.append(av)
            if len(gh)>HISTORY_SIZE: gh.pop(0)
            if len(gh)>=MIN_READINGS:
                ec=np.median(gh); et=f'Eyes:{ed}'
            else: et=f'Eyes:learning({len(gh)}/{MIN_READINGS})'; es=0
            for i in [4,133,362]: cv2.circle(f,(int(fl[i].x*w),int(fl[i].y*h)),3,(0,255,255),-1)
            for i in [468,473]: cv2.circle(f,(int(fl[i].x*w),int(fl[i].y*h)),4,(255,255,0),-1)

        if pr.pose_landmarks and len(pr.pose_landmarks)>0:
            pl=pr.pose_landmarks[0]
            bd,bs=detect_body(pl,w,h); bt=f'Body:{bd}'
            for i in [11,12]: cv2.circle(f,(int(pl[i].x*w),int(pl[i].y*h)),6,(0,165,255),-1)
            cv2.line(f,(int(pl[11].x*w),int(pl[11].y*h)),(int(pl[12].x*w),int(pl[12].y*h)),(0,165,255),2)

        total=min(100,hs+es+bs)
        def sc(s,mx):
            if s==0:return(0,180,0)
            elif s<mx*0.4:return(0,180,200)
            else:return(0,0,200)

        for i,(tx2,s,mx) in enumerate([(ht,hs,HEAD_WEIGHT),(et,es,EYE_WEIGHT),(bt,bs,BODY_WEIGHT)]):
            y=i*28;c=sc(s,mx)
            cv2.rectangle(f,(0,y),(w,y+26),c,-1)
            cv2.putText(f,f'{tx2} [{s}/{mx}]',(10,y+19),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),2)

        cv2.rectangle(f,(0,h-60),(w,h),(30,30,30),-1)
        bw=int((total/100)*(w-20))
        if total<MILD_THRESHOLD: bc=(0,200,0);v='ALL CLEAR'
        elif total<SUSPICIOUS_THRESHOLD: bc=(0,200,200);v='MILD WARNING'
        elif total<HIGH_THRESHOLD: bc=(0,100,255);v='SUSPICIOUS!'
        else: bc=(0,0,255);v='HIGH ALERT!'
        cv2.rectangle(f,(10,h-55),(10+bw,h-35),bc,-1)
        cv2.rectangle(f,(10,h-55),(w-10,h-35),(100,100,100),1)
        cv2.putText(f,f'Score:{total}/100-{v}',(10,h-10),cv2.FONT_HERSHEY_SIMPLEX,0.7,bc,2)
        el=time.time()-t0;fps=fc/el if el>0 else 0
        cv2.putText(f,f'FPS:{fps:.0f}',(w-80,h-10),cv2.FONT_HERSHEY_SIMPLEX,0.4,(150,150,150),1)
        cv2.imshow('ExamGuard Combined (q=quit)',f)
        if cv2.waitKey(1)&0xFF==ord('q'): break

    cam.release(); cv2.destroyAllWindows()
    print(f'Done! {el:.0f}s, {fps:.0f} FPS')

Running! Press q to quit.
Done! 81s, 21 FPS
